# Spotify Data Analysis 🎵🎶
By Sabrina Aezaz

## Introduction
I wanted to explore how product decisions get shaped by real user feedback, using an app I actually use every day. I pulled Spotify's general app reviews from the Google Play Store and ran them through sentiment scoring and theme analysis to find a concrete problem worth designing a fix for.

The clearest pattern in the negative reviews was reliability: crashes, freezes, and failures to load account for over a third of all negative reviews, well ahead of any other complaint category. That's the problem this notebook builds toward: not a missing feature, but people trying to use the app, having it simply not work, and giving up (often after multiple reinstalls) with no clear path forward.

## Data Collection & Cleaning

In [7]:
import pandas as pd
review_df = pd.read_excel("Spotify_Review_Data.xlsx")
review_df

,Name,Date,Review_Text,Helpful_Count
0,Sherika Johnson,August 13,audio keeps glitching out..,0
1,Brian Martinez,August 12,Continuously uses up a lot of space on the TV ...,0
2,Jared Penner,August 11,amazing!!!¡!!!!!!!!,0
3,Gelson Mwale,August 11,App is generally good. The search function can...,0
4,Vidhyavarshini V,August 11,good,0
...,...,...,...,...
572,Gedion Bocho,August 6,It is good but when I was installing it it too...,0
573,Family Seyedi,August 5,luv it,0
574,YRRAL REMMUK,August 4,Spotify is amazing for listening to music,24
575,Raju Pawar,August 3,best app for singers and love breakup,0


As you can see, we have the columns "name", "date", "review", and the "helpful_count"

### Sentiment Analysis

In [41]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

review_df['sentiment_score'] = review_df['Review_Text'].apply(
    lambda text: analyzer.polarity_scores(str(text))['compound']
)

review_df['sentiment_label'] = review_df['sentiment_score'].apply(
    lambda score: 'positive' if score > 0 else 'negative' if score < 0 else 'neutral'
)

review_df['sentiment_label'].value_counts()

sentiment_label
positive    368
neutral     132
negative     77
Name: count, dtype: int64

64% positive reviews, 23% neutral, 13% negative, which matches typical app-review skew. 

**Next,** I will filter the review data to only show us the 77 negative reviews as we're looking to improve features for Spotify, and the positive/neutral reviews will most likely not have as strong or clear of an indication of what really needs to be fixed.

The rest of the analysis will also be predominately based upon these 77 reviews.

In [40]:
negative_df = review_df[review_df['sentiment_label'] == 'negative'].copy()
for review in negative_df['Review_Text']:
    print(review)
    print('---')

app randomly starting to crash out and show black and green lines flash across my screen with a very loud high pitch white noise sound
---
having ads for podcasts on premium is lame
---
no download opsoun
---
This app is soooooooooooooo bad on TV it is so glitched
---
The app has changed and no longer acts as a live service when playing
---
so freaking laggy I can't listen to music
---
TV app is awful
---
so hard to open my account
---
I've been having problems with the website it's not just the app that is broken.
---
extremely limited application you cannot even clear out your queue or any basic functionality
---
latest update ruined the app
---
Awful! TV version is practically impossible to close. Have to use phone to tell it to stop casting even though I'm not casting. Needs a goddam Stop button!!!!!!!!!!
---
bad
---
Buggy and obvious stability issues - even with paid subscription. unprofessional - embarrassing for paid subscription.
---
They removed moon button/screen darken :(
--

## Theme Analysis

### Methodology

For the theme-tagging methodology, I'll be essentially attempting to categorize each review using using a manually defined-list of popular keywords per theme, a review can possibly match to more than one theme, and anything unmatched will fall into "uncategorized", which could include negative reviews that just say "horrible" or "bad", reviews that don't really indicate a specific issue. 

For the list, I'd also like to note that I manually skimmed through the reviews while inputting these keywords to ensure I'm including user typos and phrasing variations.

In [43]:
theme_keywords = {
    "Crashes/Freezes/Won't Load": ["crash", "crashes", "freeze", "freezes", "won't load",
                                   "not working", "bug", "bugs", "froze",
                                    "glitch", "glitches", "error", "splash screen",
                                    "frozen", "stuck", "loading",
                                    "open", "slow", "install", "download", "reinstalling"],

    "Login/Account Issues": ["login", "account", "sign in", "password", "register",
                             "sign-in", "sign up", "sign-up", "log"],

    "Ads/Pricing Complaints": ["ads", "adds", "advertisements", "pricing", "subscription",
                               "expensive", "cost", "premium", "free version", "paywall"],

    "Missing/Removed Features": ["missing", "removed", "feature", "functionality", "option",
                                 "update", "discontinued", "gone",
                                 "no download", "no lyrics", "no shuffle",
                                  "changed", "no offline mode"],

    "Playback/Streaming Issues": ["playback", "playing", "player", "streaming", "buffering",
                                   "lag", "laggy", "lags", "skip",
                                   "audio quality", "sound", "volume", "stutter",
                                   "disconnect", "drop out", "pause", "pauses", "stuck"],

    "Casting/Chromecast Issues": ["casting", "chromecast", "airplay", "spotify connect",
                                  "speaker", "device", "stream to tv", "stream to speaker"],

    "Network/Connectivity Issues": ["network", "connectivity", "wifi", "cellular", "data",
                                  "connection", "offline", "signal", "disconnect",
                                  "doesn't connect", "no connection", "error"],
}

def tag_themes(text):
    text = text.lower()
    matched_themes = [theme for theme, keywords in theme_keywords.items() 
                      if any(kw in text for kw in keywords)]
    return matched_themes if matched_themes else ["Unclassified"]

negative_df['themes'] = negative_df['Review_Text'].astype(str).apply(tag_themes)

### Results

In [ ]:
from collections import Counter

all_themes = [theme for themes in negative_df['themes'] for theme in themes]
theme_counts = Counter(all_themes)

for theme, count in theme_counts.most_common():
    pct = count / len(negative_df) * 100
    print(f"{theme}: {count} of {len(negative_df)} reviews ({pct:.0f}%)")

Unclassified: 29 of 77 reviews (38%)
Crashes/Freezes/Won't Load: 24 of 77 reviews (31%)
Playback/Streaming Issues: 10 of 77 reviews (13%)
Ads/Pricing Complaints: 10 of 77 reviews (13%)
Missing/Removed Features: 9 of 77 reviews (12%)
Login/Account Issues: 8 of 77 reviews (10%)
Network/Connectivity Issues: 4 of 77 reviews (5%)
Casting/Chromecast Issues: 3 of 77 reviews (4%)


`Crashes/Freezes/Won't Load` is the largest theme of $31\%$ (other than `Unclassified`), meaningfully ahead of the next two `Playback/Streaming Issues` and `Ads/Pricing Complaints` both having $13\%$. 

As it's the theme with the largest percentage, meaning it's impact the most amount of users, I will be focusing in that in the remaining part of this notebook

But first, let me briefly check the `Unclassified` category to make sure we're not missing anything important.

In [47]:
crash_reviews = negative_df[negative_df['themes'].apply(lambda t: "Unclassified" in t)]
crash_reviews['Review_Text']

for review in crash_reviews['Review_Text']:
    print(review)
    print('---')

TV app is awful
---
I've been having problems with the website it's not just the app that is broken.
---
bad
---
hangs a lot and is terrible
---
not having an experience sorry
---
it sucks
---
When exiting the app it still plays music even when properly exiting the app. I should be surprised that this hasn't been fixed but I am not because Spotify is more worried about the bottom line.
---
I'm sorry but it's bad
---
the app is fine but there's one problem
---
Not having a kid mode or content restrictions in 2026 is unacceptable
---
this is very bad
---
this is FRICKIN SICK!! if you need some jams well I mean Spotify's got you covered
---
what's wrong with vioume
---
add Google TV this Spotify wasn't good enough like in the Android smartphone appsno details lyrics and other links
---
it's soon good but some ad made me annoying
---
Very very scary ;)
---
very bad
---
it's so good it made my neck hurt
---
useless app mc-bc
---
it is the worst
---
not trust you
---
this app supports that a

Based upon a brief skim of the reviews, it appears that majority of the `Unclassified` reviews are either not specific or about something non-feature related, though a few reviews mention some bugs with playing music or trying to load the app on different devices. 

### Deep Dive: Crashes & Freezes

I will be pulling the reviews from the 27 crash-themed reviews to find a pattern within the pattern and potentially come up with a solution

In [45]:
crash_reviews = negative_df[negative_df['themes'].apply(lambda t: "Crashes/Freezes/Won't Load" in t)]
crash_reviews['Review_Text']

for review in crash_reviews['Review_Text']:
    print(review)
    print('---')

app randomly starting to crash out and show black and green lines flash across my screen with a very loud high pitch white noise sound
---
no download opsoun
---
This app is soooooooooooooo bad on TV it is so glitched
---
so hard to open my account
---
Buggy and obvious stability issues - even with paid subscription. unprofessional - embarrassing for paid subscription.
---
error terus
---
tv version is pathetic. screen keeps flicking.fix that bug.
---
buged out on my tcl tv says check Internet connection i cancel sub
---
I think you need some updates your app is slow to come on sometimes it cuts out on music like The feeds messed up you need to get rid of the Kinks in this app or I'm going back toYouTube music
---
App never loads beyond the splash screen on my Google TV Streamer. Not even reinstalling resolves the issue. I can cast to it but if I can't access my playlist from the TV I probably won't use it.
---
Full of bugs. Can't play pause sometimes
---
This is so good. But it's not 

## Problem & Solution

$31\%$ of negative reviews for Spotify on the Google Play Store cite crashing, freezing, or general failure to load the app. As seen in the reviews above, many users struggle through multiple installs or reinstalls, or hit a broken app with no way to resolve it and no clear next step.

A UI redesign can't fix the underlying crash bug itself, that's mainly an engineering problem, not a design one. What design *can* fix is what happens to the user in the moment of failure. Right now, that moment is a dead end: a frozen or blank screen with no guidance, so the only outlet left for a frustrated user is to leave a public 1-star review instead of getting help or having the issue logged.

My solution targets that moment directly: instead of a dead end, the user gets a fallback screen with a quick retry option, plus a path to three deeper recovery actions if a simple retry doesn't fix it: Clear Cache, Report a Problem, and Check Status.

### 1. Timeout / Fallback Screen
If the splash/loading screen hasn't resolved within 8–10 seconds, it's replaced with an actionable screen instead of hanging indefinitely: "Something went wrong, have another go?" with two options. **Try Again** covers the simple case where it was just a slow load.

![Alternative text](Error_page.jpg)

**More Options** reveals three deeper recovery actions: Clear Cache, Report a Problem, and Check Status, each covered below.

![Alternative text](More_options.png)

### 2. Clear Cache
A single tap clears the app's locally cached data, a common, low-risk fix for an app that's stuck or won't load, since corrupted or stale cached files are a frequent cause of exactly this kind of failure. The button shows a brief "Clearing cache..." loading state before confirming success, so the user knows the action actually happened rather than wondering if anything occurred.

![Alternative text](Clear_cache.jpg)

### 3. One-Tap "Report a Problem"
From that same failure screen, a single tap submits a bug report that automatically captures device and context info in the background, no manual description required from an already-frustrated user. This closes the loop on the biggest pattern in this data: right now, the Play Store review section is functioning as Spotify's accidental bug tracker. This gives users a direct path to report the issue that's actually useful to the product team, instead of a public review that isn't.

![Alternative text](Report_problem.jpg)

### 4. Check Status
Tapping **Check Status** verifies whether the installed app version is current, covering both possible outcomes. If the app is already up to date, the user gets a quick confirmation.

![Alternative text](Check_status_No_update.jpg)

 If an update exists, they see a clear "An update is available" message with an **Update Now** button that deep-links to the Play Store listing. Either way, the user leaves with an answer and a next step, instead of guessing whether updating might fix things.

![Alternative text](Check_status_Update.jpg)